# Pandas 03 — Selecting, filtering, transforming

Every operation on a 5-row table first, then one line on the real data.

**What's in here**
1. `[]` columns, 2. `loc` / `iloc`, 3. boolean masks, 4. `rename` / `assign`,
5. vectorised arithmetic vs `apply`, 6. `map` / `np.where` / `np.select`,
7. `cut` / `qcut`, 8. `shift` / `diff` / `pct_change`, 9. `rank` / `clip` / `round`,
10. sorting and extremes, 11. `set_index` / `reset_index`, 12. `SettingWithCopyWarning`, 13. `pipe`

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

In [2]:
df = pd.DataFrame({
    "hour": [0, 6, 12, 18, 23],
    "load": [20, 25, 30, 38, 22],
    "temp": [2.0, 1.0, 8.0, 6.0, 3.0],
    "day":  ["Mon", "Mon", "Mon", "Tue", "Tue"],
}, index=["r0", "r1", "r2", "r3", "r4"])
df

,hour,load,temp,day
r0,0,20,2.0,Mon
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon
r3,18,38,6.0,Tue
r4,23,22,3.0,Tue


In [3]:
real = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
real.head(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71


## 1. `[]` — columns by name

One name gives a Series; a list of names gives a DataFrame.

In [4]:
df["load"]

r0    20
r1    25
r2    30
r3    38
r4    22
Name: load, dtype: int64

In [5]:
df[["hour", "load"]]

,hour,load
r0,0,20
r1,6,25
r2,12,30
r3,18,38
r4,23,22


## 2. `loc` — by label, `iloc` — by position

`loc[row_label, column_name]`. Slices with `loc` include both ends.

In [6]:
df.loc["r1", "load"]

25

In [7]:
df.loc["r1":"r3", ["hour", "load"]]

,hour,load
r1,6,25
r2,12,30
r3,18,38


`iloc[row_position, column_position]`. Slices with `iloc` exclude the end.

In [8]:
df.iloc[1, 1]

25

In [9]:
df.iloc[1:3, 0:2]

,hour,load
r1,6,25
r2,12,30


`iloc[1:3]` gave rows r1, r2 (positions 1 and 2), while `loc["r1":"r3"]` gave r1, r2, r3.
`at` / `iat` are the fast single-cell versions.

In [10]:
print(df.at["r2", "temp"], df.iat[2, 2])

8.0 8.0


## 3. Boolean masks

A comparison gives True/False per row; passing it to `df[...]` keeps the True rows.

In [11]:
df["load"] > 24

r0    False
r1     True
r2     True
r3     True
r4    False
Name: load, dtype: bool

In [12]:
df[df["load"] > 24]

,hour,load,temp,day
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon
r3,18,38,6.0,Tue


Combine with `&`, `|`, `~`; every comparison in parentheses.

In [13]:
df[(df["load"] > 24) & (df["day"] == "Mon")]

,hour,load,temp,day
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon


In [14]:
df[~(df["day"] == "Mon")]

,hour,load,temp,day
r3,18,38,6.0,Tue
r4,23,22,3.0,Tue


**Pitfall:** `a < x < b` does not work on a Series. Use `between` or two comparisons.

In [15]:
try:
    df[20 < df["load"] < 30]
except ValueError as e:
    print("ValueError:", str(e)[:60])
df[df["load"].between(20, 30)]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.boo


,hour,load,temp,day
r0,0,20,2.0,Mon
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon
r4,23,22,3.0,Tue


In [16]:
df[df["day"].isin(["Tue"])]

,hour,load,temp,day
r3,18,38,6.0,Tue
r4,23,22,3.0,Tue


`query` writes the same condition as a string.

In [17]:
df.query("load > 24 and day == 'Mon'")

,hour,load,temp,day
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon


In [18]:
df[df["day"].str.contains("on")]

,hour,load,temp,day
r0,0,20,2.0,Mon
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon


`select_dtypes` picks columns by type.

In [19]:
df.select_dtypes("number")

,hour,load,temp
r0,0,20,2.0
r1,6,25,1.0
r2,12,30,8.0
r3,18,38,6.0
r4,23,22,3.0


In [20]:
real[(real["time"].dt.hour == 18) & (real["temp_c"] < 0)].head(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
114,2022-01-05 18:00:00+00:00,39331.8,-0.02,7.34,0.0,149.58
8994,2023-01-10 18:00:00+00:00,40308.7,-0.61,2.80,0.0,224.99
9282,2023-01-22 18:00:00+00:00,37306.4,-0.17,7.39,0.0,161.28


## 4. Renaming and `assign`

`rename(columns={old: new})`. `assign(new_col=...)` returns a new frame with the column added.

In [21]:
df.rename(columns={"load": "load_mw"})

,hour,load_mw,temp,day
r0,0,20,2.0,Mon
r1,6,25,1.0,Mon
r2,12,30,8.0,Mon
r3,18,38,6.0,Tue
r4,23,22,3.0,Tue


In [22]:
df.assign(load_kw=df["load"] * 1000)

,hour,load,temp,day,load_kw
r0,0,20,2.0,Mon,20000
r1,6,25,1.0,Mon,25000
r2,12,30,8.0,Mon,30000
r3,18,38,6.0,Tue,38000
r4,23,22,3.0,Tue,22000


The original `df` is unchanged by either (they return new frames).

In [23]:
df.columns

Index(['hour', 'load', 'temp', 'day'], dtype='object')

## 5. Vectorised arithmetic beats `apply`

Column arithmetic works on the whole column at once. `apply` calls a Python function per row.

In [24]:
df["load"] * 1000

r0    20000
r1    25000
r2    30000
r3    38000
r4    22000
Name: load, dtype: int64

In [25]:
df["load"].apply(lambda x: x * 1000)     # same numbers, a Python loop underneath

r0    20000
r1    25000
r2    30000
r3    38000
r4    22000
Name: load, dtype: int64

In [26]:
%timeit real["consumption_mwh"] * 1000
%timeit real["consumption_mwh"].apply(lambda x: x * 1000)

60.9 µs ± 4.44 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


2.37 ms ± 108 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## 6. `map`, `np.where`, `np.select`

`map(dict)` for lookups (unmapped → NaN). `np.where(cond, a, b)` for two-way choices,
`np.select` for several.

In [27]:
df["day"].map({"Mon": 0, "Tue": 1})

r0    0
r1    0
r2    0
r3    1
r4    1
Name: day, dtype: int64

In [28]:
np.where(df["load"] > 24, "high", "low")

array(['low', 'high', 'high', 'high', 'low'], dtype='<U4')

In [29]:
conditions = [df["temp"] < 2, df["temp"] < 7]
choices = ["cold", "mild"]
np.select(conditions, choices, default="warm")

array(['mild', 'cold', 'warm', 'mild', 'mild'], dtype='<U4')

Rows are checked in order: r1 (1.0) is cold, r0 (2.0) is mild, r2 (8.0) falls to the default.

## 7. Binning — `cut` (fixed edges) vs `qcut` (quantiles)

In [30]:
temps = pd.Series([-3, 1, 5, 9, 14, 22])
pd.cut(temps, bins=[-10, 0, 10, 30])

0    (-10, 0]
1     (0, 10]
2     (0, 10]
3     (0, 10]
4    (10, 30]
5    (10, 30]
dtype: category
Categories (3, interval[int64, right]): [(-10, 0] < (0, 10] < (10, 30]]

In [31]:
pd.cut(temps, bins=[-10, 0, 10, 30], labels=["freezing", "cold", "mild"])

0    freezing
1        cold
2        cold
3        cold
4        mild
5        mild
dtype: category
Categories (3, object): ['freezing' < 'cold' < 'mild']

In [32]:
pd.qcut(temps, q=3, labels=["low", "mid", "high"])

0     low
1     low
2     mid
3     mid
4    high
5    high
dtype: category
Categories (3, object): ['low' < 'mid' < 'high']

`cut` uses the edges you give; `qcut` chooses edges so each bin has the same count (2 each here).

## 8. `shift`, `diff`, `pct_change` — mind the sign

`shift(1)` = previous row (the past). `shift(-1)` = next row (the future).

In [33]:
pd.DataFrame({
    "load": df["load"],
    "shift(1)": df["load"].shift(1),
    "shift(-1)": df["load"].shift(-1),
    "diff()": df["load"].diff(),
    "pct_change()": df["load"].pct_change().round(3),
})

,load,shift(1),shift(-1),diff(),pct_change()
r0,20,NaN,25.0,NaN,NaN
r1,25,20.0,30.0,5.0,0.250
r2,30,25.0,38.0,5.0,0.200
r3,38,30.0,22.0,8.0,0.267
r4,22,38.0,NaN,-16.0,-0.421


`diff()` = load − shift(1) (25 − 20 = 5 at r1). A feature must use positive shifts; a
target uses a negative one.

## 9. `rank`, `clip`, `round`

In [34]:
pd.DataFrame({
    "load": df["load"],
    "rank()": df["load"].rank(),
    "clip(upper=30)": df["load"].clip(upper=30),
    "temp_round": df["temp"].round(0),
})

,load,rank(),clip(upper=30),temp_round
r0,20,1.0,20,2.0
r1,25,3.0,25,1.0
r2,30,4.0,30,8.0
r3,38,5.0,30,6.0
r4,22,2.0,22,3.0


## 10. Sorting and extremes

In [35]:
df.sort_values("load", ascending=False)

,hour,load,temp,day
r3,18,38,6.0,Tue
r2,12,30,8.0,Mon
r1,6,25,1.0,Mon
r4,23,22,3.0,Tue
r0,0,20,2.0,Mon


In [36]:
df.sort_values(["day", "load"], ascending=[True, False])

,hour,load,temp,day
r2,12,30,8.0,Mon
r1,6,25,1.0,Mon
r0,0,20,2.0,Mon
r3,18,38,6.0,Tue
r4,23,22,3.0,Tue


In [37]:
df.nlargest(2, "load")

,hour,load,temp,day
r3,18,38,6.0,Tue
r2,12,30,8.0,Mon


In [38]:
print("label of max load:", df["load"].idxmax())

label of max load: r3


In [39]:
real.nlargest(3, "consumption_mwh")[["time", "consumption_mwh", "temp_c"]]

,time,consumption_mwh,temp_c
906,2022-02-07 18:00:00+00:00,40824.9,1.99
8993,2023-01-10 17:00:00+00:00,40485.4,-1.17
450,2022-01-19 18:00:00+00:00,40453.7,3.00


## 11. `set_index` / `reset_index`

`set_index` moves a column into the index (labels). `reset_index` moves it back.

In [40]:
by_hour = df.set_index("hour")
by_hour

,load,temp,day
hour,,,
0,20,2.0,Mon
6,25,1.0,Mon
12,30,8.0,Mon
18,38,6.0,Tue
23,22,3.0,Tue


In [41]:
by_hour.loc[12]

load     30
temp    8.0
day     Mon
Name: 12, dtype: object

In [42]:
by_hour.reset_index()

,hour,load,temp,day
0,0,20,2.0,Mon
1,6,25,1.0,Mon
2,12,30,8.0,Mon
3,18,38,6.0,Tue
4,23,22,3.0,Tue


## 12. `SettingWithCopyWarning` — what triggers it and the fix

Filtering then assigning (`df[mask]["col"] = ...`) writes into a temporary copy.
The original does not change.

In [43]:
w = pd.DataFrame({"a": [1, 2, 3], "flag": [0, 0, 0]})
w

,a,flag
0,1,0
1,2,0
2,3,0


In [44]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    w[w["a"] > 1]["flag"] = 1
w                         # unchanged

,a,flag
0,1,0
1,2,0
2,3,0


The fix: one `.loc` with the mask and the column.

In [45]:
w.loc[w["a"] > 1, "flag"] = 1
w

,a,flag
0,1,0
1,2,1
2,3,1


If you want a subset to work on separately, take an explicit `.copy()`.

In [46]:
sub = w[w["a"] > 1].copy()
sub["flag"] = 9
print(sub)
print()
print(w)                 # still 1s: the copy is independent

   a  flag
1  2     9
2  3     9

   a  flag
0  1     0
1  2     1
2  3     1


## 13. `pipe` — functions in a chain

`df.pipe(f)` calls `f(df)`. It keeps a sequence of steps readable.

In [47]:
def add_kw(d):
    return d.assign(load_kw=d["load"] * 1000)

def keep_monday(d):
    return d[d["day"] == "Mon"]

df.pipe(add_kw).pipe(keep_monday)

,hour,load,temp,day,load_kw
r0,0,20,2.0,Mon,20000
r1,6,25,1.0,Mon,25000
r2,12,30,8.0,Mon,30000


## Quick reference

| Want | Write |
|---|---|
| one column / several | `df["c"]`, `df[["a", "b"]]` |
| by label / by position | `df.loc[r, c]`, `df.iloc[i, j]` |
| rows by condition | `df[(df.a > 1) & (df.b == "x")]` |
| between | `df[df.a.between(lo, hi)]` |
| new column, no mutation | `df.assign(c=...)` |
| conditional column | `np.where(cond, a, b)`, `np.select([...], [...], default)` |
| bins | `pd.cut(s, bins)`, `pd.qcut(s, q)` |
| previous / next | `s.shift(1)`, `s.shift(-1)` |
| write into a subset | `df.loc[mask, "c"] = v` |
| independent subset | `df[mask].copy()` |